# WSJ 2027 - Scoutnet Participant Map

Interactive KeplerGL visualization of Swedish scout participants registered for World Scout Jamboree 2027.

**Data source:** Scoutnet API  
**Visualization:** KeplerGL with heatmaps and point layers  
**Colors:** Scout Blue (#004B87) for deltagare, Gold (#E8A838) for IST

In [1]:
# Cell 1: API Configuration
import requests
import json
import os
import pandas as pd
from collections import Counter, defaultdict
from datetime import date

# Scoutnet API credentials (Arrangemang 43490)
SCOUTNET_API_ID = "43490"
SCOUTNET_API_KEY = "REDACTED"

# WSJ 2027 age cutoffs
# Deltagare: age 14-17 at event (born 2010-2013 roughly)
# IST: age 18+ at event
WSJ_DATE = date(2027, 8, 1)

# Scoutnet API endpoint (project participants)
SCOUTNET_URL = "https://www.scoutnet.se/api/project/get/participants"

print("API configuration loaded")

API configuration loaded


In [2]:
# Cell 2: Fetch participant list from Scoutnet API

def fetch_scoutnet_participants():
    """Fetch all participants from the Scoutnet project."""
    response = requests.get(
        SCOUTNET_URL,
        auth=(SCOUTNET_API_ID, SCOUTNET_API_KEY)
    )
    response.raise_for_status()
    
    data = response.json()
    participants = data.get('participants', {})
    labels = data.get('labels', {})
    
    print(f"Fetched {len(participants)} participants from Scoutnet API")
    
    # Show fee types (registration categories)
    fee_labels = labels.get('project_fee', {})
    print(f"\nFee categories:")
    for fee_id, label in fee_labels.items():
        if label:
            count = sum(1 for p in participants.values() if str(p.get('fee_id', '')) == fee_id)
            print(f"  {fee_id}: {label} ({count} st)")
    
    # Count cancelled
    cancelled = sum(1 for p in participants.values() if p.get('cancelled'))
    print(f"\nCancelled: {cancelled}")
    print(f"Active: {len(participants) - cancelled}")
    
    return data

members_data = fetch_scoutnet_participants()

Fetched 1145 participants from Scoutnet API

Fee categories:
  27561: Deltagare direktresa ansökningsavgift (184 st)
  25702: IST egen resa  ansökningsavgift (118 st)
  25694: Deltagare rundresa ansökningsavgift (764 st)
  25697: CMT betalning 1 (21 st)
  25696: IST rundresa ansökningsavgift (52 st)
  25693: Anställd/Styrelse (1 st)

Cancelled: 29
Active: 1116


In [16]:
# Cell 3: Categorize participants (deltagare vs IST) by kår
# Fee IDs from API labels:
#   27561 = Deltagare direktresa ansökningsavgift
#   25694 = Deltagare rundresa ansökningsavgift
#   25702 = IST egen resa ansökningsavgift
#   25696 = IST rundresa ansökningsavgift
#   25697 = CMT betalning 1
#   25693 = Anställd/Styrelse

DELTAGARE_FEES = {'27561', '25694'}
IST_FEES = {'25702', '25696', '25697', '25693'}

# WSJ 2027 Poland: July 29 - August 8, 2027
WSJ_START = date(2027, 7, 29)

# Age validation using exact dates at WSJ start:
# Deltagare: must be 14-17 (14 <= age < 18)
# IST: must be 18+ (age >= 18)

def exact_age(birth_date, ref_date):
    """Calculate exact age in years at ref_date."""
    years = ref_date.year - birth_date.year
    if (ref_date.month, ref_date.day) < (birth_date.month, birth_date.day):
        years -= 1
    return years

def categorize_participants(data):
    """Parse participants and categorize into deltagare and IST by kår.
    Validates exact age at WSJ start and skips wrong-age participants."""
    karer = defaultdict(lambda: {'deltagare': 0, 'ist': 0})
    
    participants = data.get('participants', {})
    total = 0
    skipped_cancelled = 0
    skipped_wrong_age = []
    no_kar_count = 0
    
    for member_id, p in participants.items():
        # Skip cancelled registrations
        if p.get('cancelled'):
            skipped_cancelled += 1
            continue
        
        # Get kår name from primary membership (can be dict or empty list)
        membership = p.get('primary_membership_info', {})
        if isinstance(membership, dict):
            kar_name = membership.get('group_name', '')
        else:
            kar_name = ''
        
        if not kar_name:
            print(f"Person without kår: {member_id}")
            no_kar_count += 1
        
        fee_id = str(p.get('fee_id', ''))
        dob = p.get('date_of_birth', '')
        name = f"{p.get('first_name', '')} {p.get('last_name', '')}"
        
        # Parse birth date
        birth = None
        if dob:
            try:
                birth = date.fromisoformat(dob)
            except ValueError:
                pass
        
        if fee_id in DELTAGARE_FEES:
            # Validate: deltagare must be 14-17 at WSJ start
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if age < 14 or age >= 18:
                    skipped_wrong_age.append(
                        f"  DELTAGARE wrong age: {name} ({kar_name or 'ingen kår'}) "
                        f"born {dob} (age {age} at WSJ) - must be 14-17"
                    )
                    continue
            if kar_name:
                karer[kar_name]['deltagare'] += 1
            total += 1
        elif fee_id in IST_FEES:
            # Validate: IST must be 18+ at WSJ start
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if age < 18:
                    skipped_wrong_age.append(
                        f"  IST too young: {name} ({kar_name or 'ingen kår'}) "
                        f"born {dob} (age {age} at WSJ) - must be 18+"
                    )
                    continue
            if kar_name:
                karer[kar_name]['ist'] += 1
            total += 1
        else:
            # Unknown fee type - use exact age to categorize
            if birth is not None:
                age = exact_age(birth, WSJ_START)
                if 14 <= age <= 17:
                    if kar_name:
                        karer[kar_name]['deltagare'] += 1
                    total += 1
                elif age >= 18:
                    if kar_name:
                        karer[kar_name]['ist'] += 1
                    total += 1
                else:
                    skipped_wrong_age.append(
                        f"  UNKNOWN category: {name} ({kar_name or 'ingen kår'}) "
                        f"fee={fee_id} born {dob} (age {age})"
                    )
                    continue
            else:
                skipped_wrong_age.append(
                    f"  NO DOB: {name} ({kar_name or 'ingen kår'}) fee={fee_id}"
                )
                continue
    
    print(f"Processed {total} active participants across {len(karer)} kårer")
    print(f"Skipped: {skipped_cancelled} cancelled")
    print(f"Participants without kår (included in count, not on map): {no_kar_count}")
    
    if skipped_wrong_age:
        print(f"\nSkipped {len(skipped_wrong_age)} with wrong age:")
        for msg in skipped_wrong_age:
            print(msg)
    else:
        print("\nNo age validation issues found")
    
    return dict(karer)

karer_data = categorize_participants(members_data)

Person without kår: 3372969
Person without kår: 3003494
Person without kår: 3302193
Person without kår: 3306414
Person without kår: 3325404
Person without kår: 3453458
Person without kår: 3453594
Person without kår: 3293916
Person without kår: 3437342
Person without kår: 3453794
Person without kår: 3453795
Person without kår: 3453478
Person without kår: 3453479
Person without kår: 3397249
Person without kår: 3252034
Person without kår: 3421627
Person without kår: 3390406
Person without kår: 3340871
Person without kår: 3376426
Person without kår: 3319305
Person without kår: 3397255
Person without kår: 3301733
Person without kår: 3417294
Person without kår: 3324845
Person without kår: 3365596
Person without kår: 3397456
Person without kår: 3305106
Person without kår: 3146273
Person without kår: 3321737
Person without kår: 3321748
Person without kår: 3424628
Person without kår: 3272553
Person without kår: 3348629
Processed 1113 active participants across 271 kårer
Skipped: 29 cancelled
Pa

In [4]:
# Cell 4: Summary statistics
total_deltagare = sum(v['deltagare'] for v in karer_data.values())
total_ist = sum(v['ist'] for v in karer_data.values())
total_all = total_deltagare + total_ist

print(f"=== WSJ 2027 Registration Statistics ===")
print(f"Total registrerade: {total_all}")
print(f"  Deltagare (14-17): {total_deltagare}")
print(f"  IST (18+):         {total_ist}")
print(f"Antal kårer:         {len(karer_data)}")
print(f"\nMax deltagare per kår: {max(v['deltagare'] for v in karer_data.values())}")
print(f"Max IST per kår:       {max(v['ist'] for v in karer_data.values())}")

=== WSJ 2027 Registration Statistics ===
Total registrerade: 1083
  Deltagare (14-17): 908
  IST (18+):         175
Antal kårer:         271

Max deltagare per kår: 17
Max IST per kår:       8


In [5]:
# Cell 5: Top kårer by deltagare
top_deltagare = sorted(karer_data.items(), key=lambda x: x[1]['deltagare'], reverse=True)[:15]

print("=== Top 15 kårer (deltagare) ===")
for kar, data in top_deltagare:
    print(f"  {kar}: {data['deltagare']} deltagare, {data['ist']} IST")

=== Top 15 kårer (deltagare) ===
  Mölndals Scoutkår: 17 deltagare, 3 IST
  Saltsjö-Boo Scoutkår: 17 deltagare, 0 IST
  Rydebäcks Scoutkår: 16 deltagare, 1 IST
  Älvsjö Scoutkår: 16 deltagare, 1 IST
  Dalby Scoutkår: 15 deltagare, 0 IST
  Örnsbergs Scoutkår: 13 deltagare, 4 IST
  Årsta Scoutkår: 13 deltagare, 1 IST
  Scoutkåren Finn: 13 deltagare, 0 IST
  Equmenia Scout: 12 deltagare, 8 IST
  Karlsro Scoutkår: 12 deltagare, 4 IST
  Vallentuna Scoutkår: 12 deltagare, 4 IST
  Segeltorps Scoutkår: 12 deltagare, 2 IST
  Nyköpings Scoutkår: 11 deltagare, 2 IST
  Staffanstorps Scoutkår Torparna: 11 deltagare, 0 IST
  Furulunds Scoutkår: 11 deltagare, 0 IST


In [6]:
# Cell 6: Top kårer by IST
top_ist = sorted(karer_data.items(), key=lambda x: x[1]['ist'], reverse=True)[:15]

print("=== Top 15 kårer (IST) ===")
for kar, data in top_ist:
    print(f"  {kar}: {data['ist']} IST, {data['deltagare']} deltagare")

=== Top 15 kårer (IST) ===
  Equmenia Scout: 8 IST, 12 deltagare
  Karlstads Scoutkår: 6 IST, 1 deltagare
  Järlinden Scoutkår: 4 IST, 9 deltagare
  Karlsro Scoutkår: 4 IST, 12 deltagare
  Örnsbergs Scoutkår: 4 IST, 13 deltagare
  Bollstanäs Scoutkår: 4 IST, 6 deltagare
  Vallentuna Scoutkår: 4 IST, 12 deltagare
  S:t Olofs Scoutkår Västerås: 4 IST, 5 deltagare
  Umeå Scoutkår: 3 IST, 3 deltagare
  Hjärnarps Scoutkår: 3 IST, 5 deltagare
  Mölndals Scoutkår: 3 IST, 17 deltagare
  Wä Scoutkår: 3 IST, 4 deltagare
  Kullavik Sjöscoutkår: 3 IST, 0 deltagare
  Linghems Scoutkår: 3 IST, 3 deltagare
  Saltsjöbadens Sjöscoutkår: 3 IST, 1 deltagare


In [7]:
# Cell 7: Distribution histogram
print("=== Distribution of deltagare per kår ===")
deltagare_counts = [v['deltagare'] for v in karer_data.values() if v['deltagare'] > 0]
ist_counts = [v['ist'] for v in karer_data.values() if v['ist'] > 0]

for bucket in range(0, max(deltagare_counts) + 2, 2):
    count = sum(1 for d in deltagare_counts if bucket <= d < bucket + 2)
    if count > 0:
        print(f"  {bucket:2d}-{bucket+1:2d} deltagare: {'█' * count} ({count} kårer)")

print(f"\nKårer med bara IST (0 deltagare): {sum(1 for v in karer_data.values() if v['deltagare'] == 0 and v['ist'] > 0)}")
print(f"Kårer med bara deltagare (0 IST): {sum(1 for v in karer_data.values() if v['ist'] == 0 and v['deltagare'] > 0)}")

=== Distribution of deltagare per kår ===
   0- 1 deltagare: ███████████████████████████████████████████████████████████████████████████████ (79 kårer)
   2- 3 deltagare: █████████████████████████████████████████████████████████████████████████ (73 kårer)
   4- 5 deltagare: █████████████████████████████████████ (37 kårer)
   6- 7 deltagare: █████████████████████████ (25 kårer)
   8- 9 deltagare: █████████ (9 kårer)
  10-11 deltagare: ████████ (8 kårer)
  12-13 deltagare: ███████ (7 kårer)
  14-15 deltagare: █ (1 kårer)
  16-17 deltagare: ████ (4 kårer)

Kårer med bara IST (0 deltagare): 28
Kårer med bara deltagare (0 IST): 167


In [8]:
# Cell 8: Geocoding with cache + manual corrections
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

CACHE_FILE = '/config/notebooks/wsj27/scoutkar_geocode_cache.json'

# Manual corrections for kårer that geocode to wrong locations
MANUAL_CORRECTIONS = {
    'Landeryd Scoutkår': (58.3725, 15.7001),  # Tallholmsvägen 62, Linköping
    'Staffanstorps Scoutkår': (55.6419, 13.2063),
    'Gustaf Vasa-Bredängs Scoutkår': (59.2941, 17.9383),
    'Mälarscouternas Scoutkår': (59.3326, 18.0649),
    'Sköndals Scoutkår': (59.2571, 18.1071),
    'Blackebergs Scoutkår': (59.3500, 17.8833),
    'S:t Görans Scoutkår': (59.3333, 18.0333),
    'Hässelby Strands Scoutkår': (59.3667, 17.8333),
    # Alternate name variants from API
    'Staffanstorps Scoutkår Torparna': (55.6419, 13.2063),
    'Scoutkåren Gustaf Vasa-Bredäng': (59.2941, 17.9383),
    'Husie-Hohögs Scoutkår': (55.5833, 13.1000),
    'Trekungakåren': (57.7089, 11.9746),
    'Scoutkåren Vikingen-Ekeby': (59.3667, 16.5000),
    'Näsby-Österslövs scoutkår': (56.1500, 14.1500),
    'Scoutkåren Mälarscouterna': (59.3326, 18.0649),
    'Sjöscoutkåren Drakarna': (59.3293, 18.0686),
    'Scoutkåren Vikingarna': (59.3293, 18.0686),
    'Drottningstaden Scoutkår': (58.5372, 13.1500),
    'Östra Eneby-Haga Scoutkår': (58.6167, 16.1833),
    'Eldsberga-Tönnersjö Scoutkår': (56.7167, 12.9500),
    'Tornugglan Scoutkår': (55.7119, 13.2283),
    'Lutherska Missionskyrkans EFS Scout (Salt) Toleredskyrkans Scoutkår': (57.7167, 11.9333),
    # Geocode corrections (wrong location from Nominatim)
    'Wä Scoutkår': (56.0167, 14.2167),  # Wä, Kristianstad
    'Scoutkåren Peter Momma': (59.2000, 17.8500),  # Stockholm
    'Sjöscoutkåren S:t Göran': (59.3333, 18.0333),  # Stockholm
    'Blackebergs sjöscoutkår': (59.3500, 17.8833),  # Stockholm
    'Sköndals Land- och Sjöscoutkår': (59.2571, 18.1071),  # Sköndal, Stockholm
    'Malå Equmenia': (65.1833, 18.7500),  # Malå, Västerbotten
    'Turinge Scoutkår': (59.1500, 17.5667),  # Turinge, Södertälje
}

WEBSEARCH_LOCATIONS = {
    'Annestorpsdalens Scoutkår': (57.6523, 12.0712),
    'Brobergska Scoutkår': (59.3293, 18.0686),
    'S:t Örjans Scoutkår': (57.7210, 12.9401),
    'Finns Scoutkår': (55.7047, 13.1910),
    'Tornugglans Scoutkår': (55.7119, 13.2283),
    'Redbergslids Scoutkår': (57.7089, 12.0000),
    'Göta Lejons Scoutkår': (57.7089, 11.9746),
    'Jägarna-Lundhagskyrkans Scoutkår': (59.8607, 17.6253),
    'Vikingarna Scoutkår': (59.3293, 18.0686),
    'Drakarna Scoutkår': (59.3293, 18.0686),
    'Equmeniascout Lerum': (57.7700, 12.2692),
    'Södermöre Scoutkår': (56.6833, 16.3667),
    'Vikingen-Ekeby Scoutkår': (59.3667, 16.5000),
    'Peter Momma Scoutkår': (59.2000, 17.8500),
    'Drottens Scoutkår': (58.4108, 15.6214),
    'Lyckebokyrkans Scoutkår': (57.7833, 12.0500),
    'Toleredskyrkans Scoutkår': (57.7167, 11.9333),
    'Lundhagskyrkans Scoutkår': (59.8560, 17.6097),
}

# Load cache
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        geocode_cache = json.load(f)
    print(f"Loaded {len(geocode_cache)} cached geocode entries")
else:
    geocode_cache = {}
    print("No cache file found, starting fresh")

# Apply manual corrections (override cache)
for kar_name in karer_data.keys():
    if kar_name in MANUAL_CORRECTIONS:
        lat, lng = MANUAL_CORRECTIONS[kar_name]
        geocode_cache[kar_name] = {'lat': lat, 'lng': lng, 'source': 'manual'}
    elif kar_name in WEBSEARCH_LOCATIONS:
        lat, lng = WEBSEARCH_LOCATIONS[kar_name]
        geocode_cache[kar_name] = {'lat': lat, 'lng': lng, 'source': 'websearch'}

# Geocode missing kårer
geolocator = Nominatim(user_agent="wsj2027_scout_map")
new_geocodes = 0

for kar_name in karer_data.keys():
    if kar_name in geocode_cache:
        continue
    if kar_name == 'Gäst / Annat':
        continue
    
    try:
        # Normalize name for search
        search_name = kar_name
        for remove in ['Sjöscoutkår', 'Sjöscoutkåren', 'Scoutkåren ', 'Scoutkår',
                        'Kustscoutkår', 'Spanarkår', 'Distriktskår', 'scoutkår',
                        'KFUM ', 'KFUM', 'Equmenia ', 'Equmeniascout ']:
            search_name = search_name.replace(remove, '')
        search_name = search_name.strip(' -')
        
        location = geolocator.geocode(f"{search_name}, Sweden", timeout=10)
        
        if location and 55 < location.latitude < 70 and 10 < location.longitude < 25:
            geocode_cache[kar_name] = {
                'lat': location.latitude,
                'lng': location.longitude,
                'source': 'nominatim',
                'display': location.address
            }
            new_geocodes += 1
            print(f"  Geocoded: {kar_name} -> {location.latitude:.4f}, {location.longitude:.4f}")
        else:
            # Try just the last word (often the town name)
            parts = search_name.split()
            if len(parts) > 1:
                location = geolocator.geocode(f"{parts[-1]}, Sweden", timeout=10)
                time.sleep(1.1)
                if location and 55 < location.latitude < 70:
                    geocode_cache[kar_name] = {
                        'lat': location.latitude, 'lng': location.longitude,
                        'source': 'nominatim_fallback', 'display': location.address
                    }
                    new_geocodes += 1
                    print(f"  Geocoded (fallback): {kar_name} -> {location.latitude:.4f}")
                else:
                    print(f"  FAILED: {kar_name}")
                    geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'failed'}
            else:
                print(f"  FAILED: {kar_name}")
                geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'failed'}
        
        time.sleep(1.1)
        
        # Save cache periodically (every 20 geocodes)
        if new_geocodes % 20 == 0 and new_geocodes > 0:
            with open(CACHE_FILE, 'w', encoding='utf-8') as f:
                json.dump(geocode_cache, f, ensure_ascii=False, indent=2)
    except GeocoderTimedOut:
        print(f"  TIMEOUT: {kar_name}")
        geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': 'timeout'}
    except Exception as e:
        print(f"  ERROR: {kar_name}: {e}")
        geocode_cache[kar_name] = {'lat': None, 'lng': None, 'source': f'error: {e}'}

# Final cache save
with open(CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(geocode_cache, f, ensure_ascii=False, indent=2)

print(f"\nGeocoded {new_geocodes} new kårer")
print(f"Total cached: {len(geocode_cache)}")

# Build DataFrame
rows = []
failed = []
for kar_name, counts in karer_data.items():
    if kar_name == 'Gäst / Annat':
        continue
    geo = geocode_cache.get(kar_name, {})
    lat = geo.get('lat')
    lng = geo.get('lng')
    
    if lat is not None and lng is not None:
        rows.append({
            'kar': kar_name,
            'lat': lat,
            'lng': lng,
            'antal': counts['deltagare'] + counts['ist'],
            'deltagare': counts['deltagare'],
            'ist': counts['ist']
        })
    else:
        failed.append(kar_name)

df_map = pd.DataFrame(rows)
print(f"\n{len(df_map)} kårer with coordinates")
print(f"{df_map['antal'].sum()} participants mapped ({df_map['antal'].sum() / total_all * 100:.1f}%)")

if failed:
    print(f"\n{len(failed)} kårer without coordinates:")
    for f_name in failed:
        d = karer_data[f_name]
        print(f"  - {f_name} ({d['deltagare']} del, {d['ist']} ist)")

Loaded 271 cached geocode entries

Geocoded 0 new kårer
Total cached: 271

270 kårer with coordinates
1082 participants mapped (99.9%)


In [9]:
# Cell 9: KeplerGL map creation
from keplergl import KeplerGl

config = {
    'version': 'v1',
    'config': {
        'mapState': {
            'latitude': 59.0,
            'longitude': 16.0,
            'zoom': 5
        },
        'visState': {
            'layers': [
                {
                    'id': 'deltagare-heatmap',
                    'type': 'heatmap',
                    'config': {
                        'dataId': 'karer',
                        'label': 'Deltagare (heatmap)',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'weightField': {'name': 'deltagare', 'type': 'integer'},
                        'visConfig': {
                            'radius': 40,
                            'opacity': 0.5,
                            'colorRange': {
                                'name': 'Scout Blue',
                                'type': 'sequential',
                                'category': 'Custom',
                                'colors': ['#e6f2ff', '#b3d9ff', '#66b3ff', '#1a8cff', '#0066cc', '#004B87']
                            }
                        }
                    }
                },
                {
                    'id': 'ist-heatmap',
                    'type': 'heatmap',
                    'config': {
                        'dataId': 'karer',
                        'label': 'IST (heatmap)',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'weightField': {'name': 'ist', 'type': 'integer'},
                        'visConfig': {
                            'radius': 25,
                            'opacity': 0.5,
                            'colorRange': {
                                'name': 'IST Gold',
                                'type': 'sequential',
                                'category': 'Custom',
                                'colors': ['#fff8e6', '#ffe6a3', '#ffd966', '#f5c542', '#e8a838', '#cc8800']
                            }
                        }
                    }
                },
                {
                    'id': 'deltagare-points',
                    'type': 'point',
                    'config': {
                        'dataId': 'karer',
                        'label': 'Deltagare',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'color': [0, 75, 135],
                        'sizeField': {'name': 'deltagare', 'type': 'integer'},
                        'sizeScale': 'sqrt',
                        'visConfig': {
                            'radius': 10,
                            'fixedRadius': False,
                            'opacity': 0.85,
                            'outline': True,
                            'thickness': 2,
                            'strokeColor': [255, 255, 255],
                            'radiusRange': [4, 50],
                            'filled': True
                        }
                    }
                },
                {
                    'id': 'ist-points',
                    'type': 'point',
                    'config': {
                        'dataId': 'karer',
                        'label': 'IST',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'color': [232, 168, 56],
                        'sizeField': {'name': 'ist', 'type': 'integer'},
                        'sizeScale': 'sqrt',
                        'visConfig': {
                            'radius': 8,
                            'fixedRadius': False,
                            'opacity': 0.9,
                            'outline': True,
                            'thickness': 2,
                            'strokeColor': [255, 255, 255],
                            'radiusRange': [3, 30],
                            'filled': True
                        }
                    }
                }
            ]
        }
    }
}

karta = KeplerGl(height=700, data={'karer': df_map}, config=config)
karta

/usr/local/lib/python3.13/dist-packages/keplergl/keplergl.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'mapState': {'latitude': 59.0, 'longitude': 16.0, 'zoom': 5}, 'vi…

In [10]:
# Cell 10: HTML export + statistics
OUTPUT_DIR = '/config/notebooks/wsj27'

# Save notebook version
notebook_html = f'{OUTPUT_DIR}/wsj_keplergl_notebook.html'
karta.save_to_html(file_name=notebook_html)

# Create fullscreen version with CSS injection
with open(notebook_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

fullscreen_style = '''
<style>
    html, body { margin: 0; padding: 0; overflow: hidden; height: 100%; width: 100%; }
    .kepler-gl, #app, #root, .map-container { 
        position: absolute !important; 
        top: 0; left: 0; right: 0; bottom: 0; 
        width: 100vw !important; 
        height: 100vh !important; 
    }
</style>
'''

html_content = html_content.replace('<body>', '<body>' + fullscreen_style, 1)

fullscreen_html = f'{OUTPUT_DIR}/wsj_keplergl_karta.html'
with open(fullscreen_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Saved notebook HTML: {notebook_html}")
print(f"Saved fullscreen HTML: {fullscreen_html}")
print(f"\n=== Final Statistics ===")
print(f"Kårer on map: {len(df_map)}")
print(f"Total deltagare: {df_map['deltagare'].sum()}")
print(f"Total IST: {df_map['ist'].sum()}")
print(f"Total mapped: {df_map['antal'].sum()}")
print(f"Coverage: {df_map['antal'].sum() / total_all * 100:.1f}%")

Map saved to /config/notebooks/wsj27/wsj_keplergl_notebook.html!
Saved notebook HTML: /config/notebooks/wsj27/wsj_keplergl_notebook.html
Saved fullscreen HTML: /config/notebooks/wsj27/wsj_keplergl_karta.html

=== Final Statistics ===
Kårer on map: 270
Total deltagare: 907
Total IST: 175
Total mapped: 1082
Coverage: 99.9%
